## Install Dependencies

In [0]:
%pip install -r requirements.txt --upgrade

In [0]:
dbutils.library.restartPython()

## Setup Widgets for Configuration

In [ ]:
dbutils.widgets.text("catalog", "main", "Catalog Name")
dbutils.widgets.text("schema", "default", "Schema Name")
dbutils.widgets.text("hf_model_id", "", "Hugging Face Model ID")
dbutils.widgets.text("model_name", "image_model", "Model Name")
dbutils.widgets.text("model_alias", "staging", "Model stage")

## Read Configuration from Widgets

In [ ]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
hf_model_id = dbutils.widgets.get("hf_model_id")
model_name = dbutils.widgets.get("model_name")
model_stage = dbutils.widgets.get("model_alias")

registered_model_name = f"{catalog}.{schema}.{model_name}"

print(f"catalog: {catalog}")
print(f"schema: {schema}")
print(f"registered model: {registered_model_name}")
print(f"model stage: {model_stage}")
print(f"Hugging face model id: {hf_model_id}")

## Download Model from HuggingFace

In [0]:
from huggingface_hub import snapshot_download
import os
import torch
import torchvision

model_path = hf_model_id #"Qwen/Qwen-Image-Edit-2509"
model_cache_path = f"/local_disk0/models_cache/{model_path}"
download_models = True

if download_models:
    os.environ["HF_HOME"] = "/local_disk0/hf"
    # os.environ["HF_TOKEN"] = dbutils.secrets.get(
    #     scope="shj_scope", key="hf_secret"
    # )
    
    snapshot_location = snapshot_download(
        repo_id=model_path,
        local_dir=model_cache_path,
        ignore_patterns="*.pth"
    )
else:
    snapshot_location = model_cache_path

snapshot_location

## Create Inference Config

In [0]:
import yaml

inference_config = {
    "model_type": "qwen_image_edit",
    "use_quantization": False
}

with open('inference_config.yml', 'w') as f:
    yaml.dump(inference_config, f)

## Prepare Input Examples and Signature

In [0]:
from mlflow.models.signature import infer_signature
from PIL import Image
import io
import base64
import pandas as pd

def pillow_image_to_base64_string(img):
    buffered = io.BytesIO()
    img.save(buffered, format="PNG")
    return base64.b64encode(buffered.getvalue()).decode("utf-8")

# Load example images
image1 = Image.open("sample_images/01_ice-castle-image.png").convert("RGB")
image2 = Image.open("sample_images/02_brown-bear-image.png").convert("RGB")

# Convert to base64
image1_base64 = pillow_image_to_base64_string(image1)
image2_base64 = pillow_image_to_base64_string(image2)

# Create input example
input_example = pd.DataFrame().from_records([{
    "image1": image1_base64,
    "image2": image2_base64,
    "prompt": "The magician emperor bear is standing in front of castle with a diamond topped septar in his hand. Keep a snowing background and a blue sky."
}])

# Define parameters
params = {
    "num_inference_steps": 40,
    "true_cfg_scale": 4.0,
    "guidance_scale": 1.0,
    "negative_prompt": " ",
    "num_images_per_prompt": 1,
    "seed": 0
}

# Create output example
output_example = pd.DataFrame().from_records([{
    "output_image": "this is an example base64 output image"
}])

signature = infer_signature(input_example, output_example, params)
print(signature)

## Log Model to MLflow

In [0]:
import mlflow
from pkg_resources import get_distribution

ds_model_path = os.path.join(os.getcwd(), "imagen_model.py")
config_path = os.path.join(os.getcwd(), "inference_config.yml")

with mlflow.start_run():
    model_info = mlflow.pyfunc.log_model(
        name="model",
        python_model=ds_model_path,
        model_config=config_path,
        artifacts={"model_path": model_cache_path},
        # code_paths=[ds_model_path],
        # input_example=input_example,
        signature=signature,
        pip_requirements=[
            f"torch=={get_distribution('torch').version}",
            f"torchvision=={get_distribution('torchvision').version}",
            "diffusers",
            "transformers",
            "accelerate",
            f"mlflow[databricks]=={get_distribution('mlflow').version}"
        ]
    )


model_info.model_uri

## Test the Logged Model

In [0]:
%sh
nvidia-smi

In [0]:
import mlflow
import pandas as pd
from PIL import Image
import io
import base64

def pillow_image_to_base64_string(img):
    buffered = io.BytesIO()
    img.save(buffered, format="PNG")
    return base64.b64encode(buffered.getvalue()).decode("utf-8")

def base64_string_to_pillow_image(base64_str):
    return Image.open(io.BytesIO(base64.decodebytes(bytes(base64_str, "utf-8"))))

# Load example images
image1 = Image.open("sample_images/01_ice-castle-image.png").convert("RGB")
image2 = Image.open("sample_images/02_brown-bear-image.png").convert("RGB")

# Convert to base64
image1_base64 = pillow_image_to_base64_string(image1)
image2_base64 = pillow_image_to_base64_string(image2)

# Create input
input_example = pd.DataFrame().from_records([{
    "image1": image1_base64,
    "image2": image2_base64,
    "prompt": "The magician emperor bear is standing in front of castle with a diamond topped septar in his hand. Keep a snowing background and a blue sky. Do not include any bookmarks"
}])

In [0]:
# Load the model (replace with your actual model URI from above)
model_uri = model_info.model_uri #'models:/m-7451867a46dc441e9beec6889e1193af'
loaded_model = mlflow.pyfunc.load_model(model_uri)

In [0]:
# Predict
output = loaded_model.predict(
    input_example,
    params={
        "num_inference_steps": 40,
        "true_cfg_scale": 4.0,
        "guidance_scale": 1.0,
        "negative_prompt": " ",
        "num_images_per_prompt": 1,
        "seed": 0
    }
)

# Display the output image
output_image_base64 = output.iloc[0]["output_image"]
output_image = base64_string_to_pillow_image(output_image_base64)
display(output_image)

In [0]:
# Save the output image
# output_image.save("sample_images/output_image_mlflow_test.png")
# print("Image saved at sample_images/output_image_mlflow_test.png")

## Register Model to Unity Catalog

In [0]:
mlflow.set_registry_uri("databricks-uc")
registered_model = mlflow.register_model(
    model_uri,
    registered_model_name
)

In [0]:
from mlflow import MlflowClient

mlflow_client = MlflowClient()
mlflow_client.set_registered_model_alias(registered_model_name, model_stage, registered_model.version)